# 00 Clone / Update GitHub Colab

本 notebook 用于在 Colab Free 中挂载 Google Drive，并把 GitHub 工程 clone 或更新到 `/content/drive/MyDrive/qwen3-asr`。

它重点解决两个问题：

- `git pull` 只显示 `Already up to date` 时，不容易判断当前代码到底是哪一个 commit。
- Drive 中已有旧工程目录时，后续训练 notebook 可能继续使用旧代码。

执行完成后，应看到当前 `HEAD`、最新 commit message，以及关键修复标记检查通过。

In [ ]:
# 挂载 Google Drive，并集中设置仓库参数。
# 后续 notebook 默认都使用 PROJECT_DIR 作为项目根目录。
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'qwen3-asr'
REPO_URL = 'https://github.com/cluster1900/lora-asr.git'
BRANCH = 'main'

# 默认不丢弃 Drive 仓库里的本地修改。
# 如果这个目录只是 Colab 克隆副本，并且你明确想完全对齐远端，可以改成 True。
FORCE_RESET = False

print('PROJECT_DIR =', PROJECT_DIR)
print('REPO_URL =', REPO_URL)
print('BRANCH =', BRANCH)
print('FORCE_RESET =', FORCE_RESET)

In [ ]:
# 封装命令执行函数：每条命令都会打印 stdout/stderr。
# 这样即使后续 cell 失败，也能直接看到真实错误，而不是只看到 CalledProcessError。
import subprocess


def run_cmd(args, cwd=None, check=True):
    """执行命令并返回 CompletedProcess。失败时默认抛出异常。"""
    print('\n$ ' + ' '.join(map(str, args)))
    result = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check:
        result.check_returncode()
    return result


def is_git_repo(path):
    """判断目录是否是 git 仓库。"""
    return (path / '.git').exists()

In [ ]:
# 如果项目不存在就 clone；如果已经存在且是 git 仓库，就更新到远端 main。
# 注意：FORCE_RESET=False 时不会覆盖本地未提交修改。
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if PROJECT_DIR.exists() and is_git_repo(PROJECT_DIR):
    print('检测到已有 git 仓库，开始更新。')
    remotes = run_cmd(['git', 'remote'], cwd=PROJECT_DIR, check=False).stdout.split()
    if 'origin' not in remotes:
        run_cmd(['git', 'remote', 'add', 'origin', REPO_URL], cwd=PROJECT_DIR)
    else:
        run_cmd(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=PROJECT_DIR)

    run_cmd(['git', 'remote', '-v'], cwd=PROJECT_DIR)
    run_cmd(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_DIR)

    print('\n本地 HEAD：')
    run_cmd(['git', 'log', '-1', '--oneline', 'HEAD'], cwd=PROJECT_DIR)
    print('\n远端 origin/main：')
    run_cmd(['git', 'log', '-1', '--oneline', f'origin/{BRANCH}'], cwd=PROJECT_DIR)

    if FORCE_RESET:
        print('\nFORCE_RESET=True：会丢弃 Drive 仓库中的本地未提交修改。')
        run_cmd(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=PROJECT_DIR)
    else:
        run_cmd(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)
elif PROJECT_DIR.exists():
    entries = list(PROJECT_DIR.iterdir())
    if entries:
        raise RuntimeError(f'{PROJECT_DIR} 已存在但不是 git 仓库。请先确认是否要改名、备份或删除。')
    print('检测到空目录，直接 clone 到该目录。')
    run_cmd(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)])
else:
    print('未检测到项目目录，开始 clone。')
    run_cmd(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)])

In [ ]:
# 打印当前代码版本。
# 如果这里已经是最新 commit，再看到 git pull 的 Already up to date 就是正常现象。
import os

os.chdir(PROJECT_DIR)
print('当前工作目录:', Path.cwd())

short_head = run_cmd(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR).stdout.strip()
last_commit = run_cmd(['git', 'log', '-1', '--oneline'], cwd=PROJECT_DIR).stdout.strip()
status = run_cmd(['git', 'status', '--short'], cwd=PROJECT_DIR).stdout.strip()

print('\n当前代码版本确认：')
print('HEAD =', short_head)
print('COMMIT =', last_commit)
print('STATUS =', status if status else 'clean')

In [ ]:
# 检查关键文件和关键修复标记。
# 这些检查不是训练本身，只用于确认 Drive 里的代码已经包含我们当前要跑的版本。
checks = {
    Path('train/train_qwen3_asr_lora.py'): [
        'resolve_training_model',
        'target_root_prefix',
        'training_root',
    ],
    Path('train/peft_targets.py'): [
        'root_prefix',
        'prefixed_module_name',
    ],
    Path('configs/train/qwen3_asr_lora_mvp.yaml'): [
        'gradient_checkpointing: false',
    ],
    Path('notebooks/03_train_lora_colab.ipynb'): [
        'Transformers + PEFT',
        'stderr tail',
    ],
}

failures = []
for rel_path, needles in checks.items():
    path = PROJECT_DIR / rel_path
    if not path.exists():
        failures.append(f'缺失文件: {rel_path}')
        continue
    text = path.read_text(encoding='utf-8')
    for needle in needles:
        if needle not in text:
            failures.append(f'{rel_path} 缺失关键标记: {needle}')

if failures:
    print('\n版本验收失败：')
    for item in failures:
        print('-', item)
    raise RuntimeError('Drive 中的工程代码还不是预期版本，请先更新或重新 clone。')

print('\n版本验收通过，可以继续执行 01/02/03 notebook。')